In [1]:
# This is boilerplate code to correctly setup the settings to for notebook.
import os, sys
for root, dirs, files in os.walk(os.getcwd()):
    # print(root)

    if "pion-argon-xs-analysis" in root.split("/"):
        pypath = root.split("pion-argon-xs-analysis")[0] + "/pion-argon-xs-analysis/analysis"
        print(pypath)
        break

sys.path.insert(0, pypath)

from python.analysis.NotebookUtils import init_notebook
%init_notebook

/home/bhuller/xs_analysis//pion-argon-xs-analysis/analysis
/home/bhuller/xs_analysis//pion-argon-xs-analysis/analysis
env: PYTHONPATH=/home/bhuller/xs_analysis//pion-argon-xs-analysis/analysis


In [2]:
from python.analysis import cross_section
from apps import cex_analysis_input

In [3]:
cfg = cross_section.ApplicationArguments.ResolveConfig(cross_section.LoadConfiguration("/home/bhuller/work/analysis_main/cex_analysis_2GeV_config.json"))

In [9]:
cfg.ntuple_files["mc"]

[FileDescriptor(file='/data/dune/common/PDSPAnalyzer_Ntuples/PDSPProd4a_MC_2GeV_sce_datadriven_ntuple_v09_81_00d01_set0.root', type='PDSPAnalyser', pmom=1),
 FileDescriptor(file='/data/dune/common/PDSPAnalyzer_Ntuples/PDSPProd4a_MC_2GeV_sce_datadriven_ntuple_v09_81_00d01_set1.root', type='PDSPAnalyser', pmom=1),
 FileDescriptor(file='/data/dune/common/PDSPAnalyzer_Ntuples/PDSPProd4a_MC_2GeV_sce_datadriven_ntuple_v09_81_00d01_set2.root', type='PDSPAnalyser', pmom=1),
 FileDescriptor(file='/data/dune/common/PDSPAnalyzer_Ntuples/PDSPProd4a_MC_2GeV_sce_datadriven_ntuple_v09_81_00d01_set3.root', type='PDSPAnalyser', pmom=1),
 FileDescriptor(file='/data/dune/common/PDSPAnalyzer_Ntuples/PDSPProd4a_MC_2GeV_reco1_sce_datadriven_v1_ntuple_v09_41_00_03.root', type='PDSPAnalyser', pmom=2)]

In [4]:
mc = cross_section.AnalysisInput.FromFile(cfg.analysis_input["mc"])

In [13]:
import dataclasses

mc_out = cross_section.LoadObject("/home/bhuller/work/analysis_main/output.dill")
for i in mc_out:
    setattr(i["cheated"], "region_id", None)
# req_fields = [f.name for f in cross_section.AnalysisInput() if f.init is True]

In [ ]:
ais = [mc_out[0]["cheated"], mc_out[1]["cheated"]]

cross_section.AnalysisInput.req_fields()

# req_fields = [k for k, v in cross_section.AnalysisInput.__dataclass_fields__.items() if v.init is True]
# fields = cross_section.MergeOutputs([{f : getattr(a, f) for f in req_fields} for a in ais])

# for k in fields:
#     if (type(fields[k]) == list) and (all(ak.is_none(fields[k]))):
#         fields[k] = None

# cross_section.AnalysisInput(**fields)
# return croAnalysisInput(**fields)


# cross_section.AnalysisInput.Concatenate(ais)


AnalysisInput(regions=None, inclusive_process=<Array [True, True, True, True, ..., True, True, True] type='164401 * bool'>, exclusive_process={'absorption': <Array [False, False, False, ..., False, False, False] type='164401 * bool'>, 'charge_exchange': <Array [False, False, False, ..., False, False, False] type='164401 * bool'>, 'pion_production': <Array [True, True, True, False, ..., True, True, True] type='164401 * bool'>}, process_id=<Array [2, 2, 2, 0, 2, 0, 2, 2, ..., 2, 2, 2, 2, 2, 2, 2] type='164401 * int64'>, region_id=None, outside_tpc_reco=<Array [False, False, False, ..., False, False, False] type='164401 * bool'>, outside_tpc_true=<Array [False, False, False, ..., False, False, False] type='164401 * bool'>, track_length_reco=<Array [57, 58, 135, 0, 24.8, ..., 182, 131, 118, 196] type='164401 * float64'>, KE_int_reco=<Array [1.72e+03, 1.69e+03, ..., 1.67e+03, 1.4e+03] type='164401 * float64'>, KE_init_reco=<Array [1.85e+03, 1.83e+03, ..., 1.94e+03, 1.86e+03] type='164401 * 

In [32]:
import awkward as ak

all(ak.is_none(fields["regions"]))

True

In [14]:
total_events = len(mc)
total_signal = sum([sum(v) for _, v in mc.exclusive_process.items()])
print(f"{total_events=}")
print({k : sum(v) for k, v in mc.exclusive_process.items()})
print(f"{total_signal=}")
print(f"{total_events - total_signal=}")


total_events=107773
{'absorption': np.int64(10907), 'charge_exchange': np.int64(13314), 'pion_production': np.int64(83552)}
total_signal=np.int64(107773)
total_events - total_signal=np.int64(0)


In [15]:
total_inclusive = sum(mc.inclusive_process)
other = sum(~mc.inclusive_process)

print(f"{total_inclusive=}")
print(f"{other=}")


total_inclusive=np.int64(107545)
other=np.int64(228)


In [ ]:
import awkward as ak

region_id = ak.zeros_like(mc.KE_int_reco) + -1 # -1 for uncategorised

for i, (k, v) in enumerate(mc.regions.items()):
    region_id = ak.where(v, i, region_id)
region_id

<Array [2, 2, 2, 0, 0, -1, -1, ..., 0, 2, 1, 1, 1, 2] type='107773 * ?float64'>